# DBMS Lecture 2: Advanced Data Retrieval & Subqueries

## Table of Contents
1. [Recap of Lecture 1 & Supplemental Concepts](#recap-of-lecture-1-supplemental-concepts)
2. [Introduction to Threading, Concurrency, and Databases](#introduction-to-threading-concurrency-and-databases)
   - [Hands-On Lab Part 1: Sequential vs. Concurrent Execution](#hands-on-lab-part-1-sequential-vs-concurrent-execution)
3. [Advanced Database Architectures: Massively Parallel Processing (MPP)](#advanced-database-architectures-massively-parallel-processing-mpp)
4. [Leading Practices: Concurrency vs. Connection Management in Databases](#leading-practices-concurrency-vs-connection-management-in-databases)
5. [Advanced Operational Considerations: Thrashing, Contention, and Timeouts](#advanced-operational-considerations-thrashing-contention-and-timeouts)
6. [Execution Order of SQL Statements](#execution-order-of-sql-statements)
   - [Hands-On Lab Part 2: Impact of Logical Execution Order](#hands-on-lab-part-2-impact-of-logical-execution-order)
7. [Subqueries (Nested Queries)](#subqueries-nested-queries)

## Prerequisites and Environment Setup

To execute the exercises in this notebook, a Python environment with access to a MySQL database is required. Follow the instructions below based on the operating system to set up the environment.

### 1. Virtual Environment Configuration

It is recommended to use a virtual environment to isolate dependencies.

**On Linux / macOS:**
```bash
python3 -m venv venv
source venv/bin/activate
```

**On Windows:**
```bash
python -m venv venv
venv\Scripts\activate
```

### 2. Installing the Jupyter Kernel

To ensure the notebook uses the dependencies installed in the virtual environment, the environment must be registered as a Jupyter kernel.

1. Install `ipykernel` inside the activated virtual environment:
```bash
python -m pip install ipykernel
```

> [!NOTE]
> Utilizing `python -m pip` instead of the bare `pip` command ensures that the packages are installed for the specific Python interpreter currently active or targeted, avoiding conflicts when multiple Python installations exist on the system.

2. Register the kernel:
```bash
python -m ipykernel install --user --name=dbms-lecture2 --display-name "Python (DBMS Lecture 2)"
```
3. In the Jupyter Notebook interface, navigate to the menu: **Kernel** -> **Change Kernel** and select **Python (DBMS Lecture 2)**.

### 3. Database Prerequisites

A running MySQL server is required. The default configuration assumes `localhost` with user `root` and password `password`. These can be overridden using environment variables:
- `MYSQL_HOST`
- `MYSQL_USER`
- `MYSQL_PASSWORD`

## Recap of Lecture 1 & Supplemental Concepts

A succinct review of the foundational concepts established in Lecture 1, supplemented with additional details not explicitly covered in the previous material but critical for real-world application.

### Core Highlights:
- **Programmatic Connectivity**: The utilization of Python DB-API (`mysql.connector`) for establishing communication channels with database systems.
- **Security Practices**: The critical importance of neutralizing SQL injection vulnerabilities through parameterized queries rather than raw string interpolation.
- **Normalization Principles**: The progression from unnormalized datasets to highly structured relational schemas (1NF through 6NF, and BCNF) to eliminate data redundancy and modification anomalies.

### Supplemental Concepts:
- **The Cost of Normalization**: While normalization minimizes redundancy, it increases the number of tables and required `JOIN` operations. In high-throughput systems (OLTP), excessive joins can degrade performance. Consequently, controlled de-normalization is frequently employed to optimize read speeds.
- **Connection Pooling**: Production environments rarely open and close database connections per request. Instead, connection pooling is employed to reuse active channels, significantly reducing latency and database load.

## Introduction to Threading, Concurrency, and Databases

To understand query optimization fully, one must look beyond the syntax of SQL statements and consider how the database engine and application layer manage execution resources. This section explores the concepts of threading, concurrency, and parallelism, and their relevance to database systems.

### Conceptual Definitions:
- **Process**: An instance of a computer program that is being executed. It possesses its own independent memory space, system resources, and environment variables allocated by the operating system.
  - *Example*: A running instance of a web browser (e.g., Google Chrome) or a database server (e.g., MySQL) is a distinct process.
- **Thread**: The smallest unit of execution within a process. A single process can contain multiple threads.
  - *Example*: Within a web browser process, one thread may handle the user interface rendering, another may handle network requests for downloading images, and a third may handle tab state management.
- **CPU Cores**: Physical processing units on a CPU chip. Each core can read and execute program instructions independently.
- **Concurrency**: The ability of a system to handle multiple tasks at the same time by overlapping their execution (often via time-slicing on a single core).
- **Parallelism**: The simultaneous execution of multiple tasks on multiple processing units (CPU cores).

### The Relationship Between Processes and Threads:
A process acts as a container for resources, while threads are the actual execution units that perform the work.

- **Memory Space: Shared vs. Isolated**:
  - **Shared Memory (The Heap)**: All threads within the same process share the same **Heap** memory region. The heap is a large, unstructured pool of memory used for dynamic allocation (e.g., objects, large data structures). Because it is shared, data created in the heap by one thread can be accessed by others, necessitating synchronization mechanisms (such as locks) to prevent race conditions. Access to heap memory is generally slower than stack memory.
  - **Isolated Memory (The Stack)**: Each thread possesses its own private **Stack** memory and CPU **Registers** (including the program counter). The stack is a smaller, strictly organized memory region operating on a Last-In-First-Out (LIFO) basis. It stores local variables and function call frames, ensuring that one thread's local execution state does not interfere with another's. Access to stack memory is extremely fast.
- **Isolation**: Processes are isolated from one another by the operating system. If one process crashes, it does not directly affect other processes. However, because threads share the heap memory, a memory corruption or unhandled exception in one thread can potentially terminate the entire process.
- **Overhead**: Creating and switching between processes (context switching) is resource-intensive because the OS must allocate new memory and manage resources. Creating and switching between threads is much lighter and faster.

### Types of Threads:
Threads are managed at different levels, leading to two primary classifications:
- **User-Level Threads**: Managed entirely by a runtime library in user space, without the operating system's knowledge. They are extremely fast to create and switch between, but the OS cannot schedule them across multiple CPU cores.
  - *Mechanism*: The OS only sees the parent process and schedules it on a single CPU core. The user library then time-slices the user threads within that process on that single core. Consequently, they cannot achieve true hardware parallelism. Furthermore, if one user thread makes a blocking system call (like waiting for I/O), the OS perceives the entire process as blocked and suspends it, pausing all other user threads within it.
- **Kernel-Level Threads**: Managed and scheduled directly by the operating system kernel. The OS is aware of these threads and can schedule them on different CPU cores, achieving true parallelism. Most modern programming languages and database engines utilize kernel-level threads (or map user threads to kernel threads).

### CPU-Bound vs. I/O-Bound Tasks and Thread Impact:
To understand how threads are impacted, one must distinguish between two types of workloads:

- **CPU-Bound Tasks**: Tasks that require continuous CPU computation (e.g., complex mathematical operations, data processing in memory). The thread is actively executing instructions on the CPU core. Performance is limited by CPU speed and the number of available cores.
- **I/O-Bound Tasks**: Tasks that spend most of their time waiting for external input/output operations to complete (e.g., reading a file from disk, sending a network request, waiting for a database query to return results).

**How I/O Operations Impact Threads**:
When a thread initiates a synchronous (blocking) I/O operation, it cannot proceed until the external resource responds.
- **For Kernel-Level Threads**: The operating system recognizes that the thread is waiting. It moves the thread into a "blocked" state and removes it from the CPU core, allowing another thread to use that core. Once the I/O operation completes, the OS wakes the thread up and schedules it to run again. Therefore, blocking I/O does not waste CPU cycles, but it does pause the specific thread's progress.
- **For User-Level Threads**: Because the OS does not know about individual user threads, a blocking I/O call made by one user thread causes the OS to suspend the *entire process*. Consequently, all other user threads in that process are paused, even if they are ready to execute CPU tasks.

### Threads vs. CPU Cores:
Threads are software-level concepts representing a sequence of instructions. CPU Cores are the hardware execution units.
- **Standard Core**: A standard CPU core can execute instructions from only one thread at any given microsecond. Concurrency allows thousands of threads to run on a few CPU cores by rapidly switching between them (context switching). Parallelism occurs when multiple threads run on multiple cores at the exact same time.
- **Modern Core (Hyper-Threading/SMT)**: Many modern processors support **Simultaneous Multi-Threading (SMT)** or **Hyper-Threading**. This technology allows a single physical CPU core to execute instructions from **two threads** simultaneously by duplicating certain hardware components (like the program counter and registers) while sharing the main execution units. To the operating system, a physical core with hyper-threading appears as two logical cores.

### Visualizing Concurrency vs. Parallelism

#### 1. Concurrency (Single Core, Interleaved Execution)
Tasks make progress in overlapping time periods by sharing a single CPU core via time-slicing. No true simultaneous execution occurs.

| Physical Core | Time Slice 1 | Time Slice 2 | Time Slice 3 | Time Slice 4 |
| :--- | :--- | :--- | :--- | :--- |
| **Core 1** | Task 1 | Task 2 | Task 1 | Task 2 |

#### 2. Thread-Level Parallelism (Single Core with Hyper-Threading)
A single physical core duplicates certain hardware components to execute instructions from two threads simultaneously.

| Physical Core | Logical Core | Time Slice 1 | Time Slice 2 | Time Slice 3 | Time Slice 4 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Core 1** | **Logical 1** | Task 1 | Task 1 | Task 1 | Task 1 |
| | **Logical 2** | Task 2 | Task 2 | Task 2 | Task 2 |

#### 3. Core-Level Parallelism (Multi-Core, Simultaneous Execution)
Tasks execute at the exact same time on physically separate CPU cores.

| Physical Cores | Time Slice 1 | Time Slice 2 | Time Slice 3 | Time Slice 4 |
| :--- | :--- | :--- | :--- | :--- |
| **Core 1** | Task 1 | Task 1 | Task 1 | Task 1 |
| **Core 2** | Task 2 | Task 2 | Task 2 | Task 2 |

### The Python Context: The Global Interpreter Lock (GIL)

In standard Python (CPython), multi-threading behavior is heavily influenced by the **Global Interpreter Lock (GIL)**.

*What is CPython?*: CPython is the default and most widely used implementation of the Python language. It is written in C. When a Python script is executed, CPython first **compiles** the human-readable source code (`.py`) into an intermediate format called **bytecode** (`.pyc`). Then, the CPython **interpreter** (virtual machine) executes that bytecode.

- **What is the GIL?**: The GIL is a mutex (a mutual exclusion lock) that allows only **one thread** to hold control of the Python interpreter at any given instant.
- **Why does it exist?**: CPython's memory management (specifically reference counting for garbage collection) is not thread-safe. The GIL was implemented to prevent multiple threads from simultaneously accessing or modifying Python objects, avoiding race conditions and memory corruption.
- **Python Threads are Kernel Threads**: When a thread is created in Python using the `threading` module, the runtime spawns a real operating system thread (kernel thread). The OS is aware of these threads and schedules them. However, the GIL restricts their execution.

**Why CPU-Bound Tasks are Restricted vs. I/O-Bound Tasks Benefit**:
- **CPU-Bound Tasks**: A thread performing heavy calculations (e.g., complex mathematical operations) continuously executes Python bytecode. To do so, it must hold the GIL. It rarely releases the lock except at specific intervals to allow other threads to attempt acquisition. Consequently, multiple CPU-bound threads end up running sequentially (time-slicing) rather than in parallel, even on multi-core systems.
- **I/O-Bound Tasks**: A thread waiting for external operations (like a database query response or disk read) does not need to execute Python bytecode while waiting. Standard Python libraries and database drivers are designed to **release the GIL** immediately before initiating a blocking I/O operation. While Thread 1 is waiting for the database (and has released the GIL), Thread 2 can acquire the GIL and execute Python code on the CPU. This allows multiple I/O operations to be in flight simultaneously, significantly improving throughput.

*Recent Developments*: Python 3.13 introduced experimental support for running without the GIL (free-threaded Python), aiming to enable true multi-threaded parallelism for CPU-bound tasks in the future.

**Other Python Implementations**:
- **PyPy**: An implementation focused on speed, utilizing a **Just-In-Time (JIT) compiler** to execute code significantly faster than CPython for certain workloads.
  - *What is JIT?*: Just-In-Time compilation is a technique where the runtime environment compiles parts of the bytecode into native machine code *during* execution (on the fly), rather than interpreting it line-by-line. It identifies "hot spots" (frequently executed code) and optimizes them for direct hardware execution, combining the speed of compiled languages with the flexibility of interpreted ones.
  - *How it works and Example*: Once a block of code is compiled into machine code, the interpreter is bypassed for that specific block in subsequent executions. For example, in a loop that runs 1,000,000 times to calculate a sum, the interpreter might execute the first few hundred iterations. The JIT compiler detects this frequently executed loop (a "hot spot"), translates it into optimized native machine code, and executes the remaining iterations directly on the CPU at full hardware speed, bypassing the interpreter entirely for that section.
- **Jython**: An implementation written in Java that compiles Python code to Java bytecode, allowing seamless integration with Java libraries and the Java Virtual Machine (JVM).
- **IronPython**: An implementation tightly integrated with the .NET framework, written in C# and running on the Common Language Runtime (CLR).
- **MicroPython**: A lean implementation optimized to run on microcontrollers and in resource-constrained environments.

### Relevance to Database Systems:
- **Connection Pooling**: Establishing a database connection is resource-intensive (requiring network handshakes, authentication, and memory allocation). In a multi-threaded application, opening a new connection for every thread is inefficient and can overwhelm the database server.
  - *How it Works (with Taxi Fleet Analogy)*:
    1. **Initialization**: When the application starts, a connection pool manager creates a specific number of database connections (e.g., a pool of 10) and keeps them open.
       - *Analogy*: A taxi company buys 10 cars and parks them at the station, ready for passengers.
    2. **Acquisition**: When a thread needs to execute a query, it requests an available connection from the pool instead of creating a new one.
       - *Analogy*: A passenger arrives at the station and gets into the first available taxi.
    3. **Execution**: The thread uses the borrowed connection to perform the database operation.
       - *Analogy*: The taxi drives the passenger to the destination.
    4. **Release**: Once the operation is complete, the thread returns the connection to the pool. The connection remains open and becomes available for other threads.
       - *Analogy*: The passenger arrives, gets out, and the taxi returns to the station to wait for the next person.
    5. **Scale and Overflow**: If all connections are in use, requesting threads are queued until a connection is released, or the pool may create temporary new connections up to a defined maximum limit.
       - *Analogy*: If all 10 taxis are busy, new passengers must wait in a queue, or the company calls in a backup taxi.
- **Multi-Threading in Databases**: Modern database engines (like MySQL, PostgreSQL) are inherently multi-threaded or multi-process. They can process multiple incoming queries in parallel by assigning them to different threads or processes, utilizing available CPU cores.
- **Query Optimization & Resource Management**: Query optimization is not merely about reducing query execution time for a single request. It is also about reducing resource consumption (CPU, memory, I/O) so that the database can handle more concurrent queries efficiently.
  - *Concrete Database Example (Resource Exhaustion and Locking)*:
    - **The Scenario**: Consider an e-commerce database with a large `orders` table containing millions of rows.
    - **The Problematic Query**: A user executes a complex analytics query to calculate total sales per region over the last 5 years. This query lacks proper indexes, forcing the database to perform a full table scan and massive sort operations in memory, consuming 100% of the database server's CPU for 2 minutes.
    - **The System Impact**: While this query is running, thousands of online customers are trying to place new orders (typically a fast operation taking milliseconds). However, because the slow analytics query is hogging the CPU, the fast order queries are queued and cannot get execution time. Furthermore, if the analytics query holds a shared lock on the `orders` table to ensure data consistency, new order inserts (which require exclusive locks) will be blocked until the slow query completes.
    - **Result**: The entire e-commerce platform becomes slow or unresponsive for all users, demonstrating how a single unoptimized query can bottleneck a highly concurrent database system.

## Advanced Database Architectures: Massively Parallel Processing (MPP)

For extremely large datasets typical of Data Warehousing and Big Data Analytics, standard multi-threading on a single machine may not be sufficient. In such scenarios, **Massively Parallel Processing (MPP)** databases are employed.

### Core Concepts of MPP:
- **Shared-Nothing Architecture**: Unlike traditional databases where multiple processors share memory and disks, MPP databases consist of a cluster of independent servers (nodes). Each node possesses its own CPU, memory, and storage. They share no resources and communicate solely over a high-speed network interconnect.
- **Data Partitioning (Sharding)**: Data is distributed across all nodes in the cluster based on a specific distribution strategy (e.g., hash-based on a key, or round-robin). This ensures that no single node holds all the data.

### How Queries are Executed in Parallel:
1. **Query Reception**: A designated coordinator (or master) node receives the SQL query from the user.
2. **Execution Planning**: The coordinator creates a distributed execution plan, breaking the query into smaller, independent tasks.
   - *What "Breaking the Query" Means*: If a query asks to calculate the total sales from a table with 1 billion rows distributed across 10 nodes, the coordinator does not process all rows itself. Instead, it breaks the operation into:
     - **Local Tasks**: Each of the 10 worker nodes is instructed to calculate the total sales for the specific subset of data stored on its local disk (e.g., 100 million rows each).
     - **Aggregation Tasks**: The worker nodes send their partial totals back to the coordinator, which then sums the 10 partial results to produce the final answer.
3. **Task Distribution**: The coordinator sends these tasks to the worker nodes holding the relevant data.
4. **Simultaneous Execution**: All worker nodes execute their tasks in parallel on their local subset of data.
5. **Result Aggregation**: The worker nodes send their intermediate results back to the coordinator, which aggregates them and returns the final result to the user.

### Examples of MPP Databases:
- **Teradata**: An early commercial implementation of MPP database technology.
- **Greenplum**: An open-source MPP database based on PostgreSQL.
- **Cloud-Native Options**: Google BigQuery, Amazon Redshift, and Snowflake utilize variations of MPP architecture for distributed query execution.

### Hands-On Lab Part 1: Sequential vs. Concurrent Execution

The following code demonstrates the difference between sequential execution and concurrent execution. This simulates a scenario where multiple database queries are executed.

In [ ]:
import time
import threading
from concurrent.futures import (
    ThreadPoolExecutor,
)  # High-level interface for asynchronously executing functions using threads.


def simulate_db_query(query_id):
    """
    Simulates a time-consuming database query (I/O bound).
    The function pauses for 1 second to mimic network latency or database processing time.
    """
    print(f"Thread {threading.current_thread().name} starting query {query_id}")
    # Note: time.sleep is used here to simulate an I/O-bound task (waiting), not a CPU-bound task.
    # During sleep, the thread releases the GIL, allowing other threads to run.
    time.sleep(1)
    print(f"Thread {threading.current_thread().name} finished query {query_id}")
    return f"Result from query {query_id}"


# --- Sequential Execution ---
# Queries are executed one after another in a single thread.
# The total time will be the sum of all individual query times.
start_time = time.time()
print("Starting sequential execution...")
results_seq = []
for i in range(5):
    results_seq.append(simulate_db_query(i))
end_time = time.time()
print(f"Sequential execution completed in {end_time - start_time:.6f} seconds")
print(f"Sequential Results: {results_seq}\n")

# --- Concurrent Execution ---
# Queries are executed concurrently using a pool of threads.
# The ThreadPoolExecutor manages a set of worker threads.
# The total time will be close to the time of the longest single query, rather than the sum.
start_time = time.time()
print("Starting concurrent execution...")
results_con = []
# A context manager (with statement) ensures threads are cleaned up properly.
with ThreadPoolExecutor(max_workers=5) as executor:
    # A list comprehension is used here to submit tasks to the executor.
    # It iterates 5 times, submitting 'simulate_db_query' with argument 'i' each time.
    # executor.submit returns a 'Future' object representing the execution.
    futures = [executor.submit(simulate_db_query, i) for i in range(5)]
    # The result() method waits for the thread to complete and retrieves the returned value.
    results_con = [f.result() for f in futures]
end_time = time.time()
print(f"Concurrent execution completed in {end_time - start_time:.6f} seconds")
print(f"Concurrent Results: {results_con}")

## Leading Practices: Concurrency vs. Connection Management in Databases

While the previous code example demonstrates that `ThreadPoolExecutor` can hide the latency of I/O-bound tasks (like database queries or connection creation), applying this directly to database connections requires careful architectural consideration.

### 1. Dispatching Queries Concurrently
Using a thread pool to execute multiple database queries simultaneously is a standard and effective practice. While one thread is waiting for the database to return results, other threads can utilize the CPU or issue their own queries, improving overall application throughput.

### 2. The Pitfall of Concurrent Connection Creation
Although creating connections is an I/O-bound task and *can* be sped up using threads, doing so repeatedly is considered an anti-pattern in production systems:
- **Resource Waste**: Even when done concurrently, creating a connection still requires network handshakes and cryptographic calculations. Reusing an existing connection is always faster and less resource-intensive.
- **Connection Exhaustion**: Databases have a hard limit on the number of concurrent connections they can accept. Spawning many threads to create connections simultaneously can overwhelm the database, leading to rejected connections.

### The Optimal Architecture: Connection Pooling + Concurrency
The industry-standard approach combines both concepts:
1. **Connection Pool**: A pool of database connections is initialized at application startup.
2. **Thread Pool**: A thread pool (such as `ThreadPoolExecutor`) is used to handle incoming application requests.
3. **Execution**: When a thread needs to run a query, it borrows an existing, open connection from the pool, executes the query, and returns the connection to the pool.

This ensures high concurrency without the overhead or risk of repeated connection creation.

## Advanced Operational Considerations: Thrashing, Contention, and Timeouts

In production environments, managing the interaction between concurrent applications and the database requires handling failure scenarios like resource saturation and thrashing.

### Query-Level Timeouts at the Application Level
Applications can and should set **query-level timeouts**. Most database drivers and Object-Relational Mappers (ORMs) allow developers to specify a maximum duration for any single query. If the database does not return results within this specified time (e.g., 5 seconds), the application driver aborts the request and throws an exception. This prevents application threads from hanging indefinitely waiting for a stalled database.

### Database Thrashing and Contention
A critical failure state occurs when the database connection limit has *not* been reached, but the database is **thrashing** (experiencing severe resource contention).

- **The Mechanism**: The database accepts new connections because it is below its maximum limit. However, once queries are dispatched, they enter an execution queue. Because the CPU or disk I/O is already at 100% utilization, new queries receive minimal processing time.
- **Congestive Collapse**: As queries take longer to execute, application threads wait longer, leading to more concurrent requests piling up. Accepting these new requests consumes additional memory and scheduling overhead, making the database even slower. This positive feedback loop of degradation is known as congestive collapse.

### Strategies to Handle Database Overload
To prevent congestive collapse and manage database contention, several strategies are employed in system design:

1. **Circuit Breakers**: A software pattern that monitors for failures (like timeouts). If the failure rate crosses a threshold, the circuit breaker "trips" and the application immediately rejects new database requests without even attempting to connect, giving the database time to recover.
2. **Load Shedding**: The application monitors database health (e.g., queue length or response time). If the database becomes slow, the application actively rejects low-priority requests at the entry point to protect the core system.
3. **Proper Connection Pool Sizing**: Instead of setting the connection pool maximum to a very high number, it should be set to the maximum number of concurrent queries the database hardware can actually execute efficiently without thrashing (often related to the number of CPU cores).

## Execution Order of SQL Statements

A common misconception among beginners is that SQL queries are executed in the order they are written (lexical order). However, the database engine processes clauses in a specific logical order to optimize data retrieval and manipulation.

### Lexical Order (How SQL is written):
1. `SELECT`
2. `FROM`
3. `WHERE`
4. `GROUP BY`
5. `HAVING`
6. `ORDER BY`
7. `LIMIT`

### Logical Execution Order (How the engine processes it):
1. **`FROM` / `JOIN`**: The database identifies the source tables and performs any joins to create the base dataset.
2. **`WHERE`**: Rows are filtered based on the specified conditions. This is the earliest point to reduce data volume.
3. **`GROUP BY`**: The remaining rows are grouped based on the specified columns.
4. **`HAVING`**: Grouped data is filtered based on conditions applied to aggregate functions.
5. **`SELECT`**: The engine determines which columns to return. This is where aliases are created.
6. **`ORDER BY`**: The final result set is sorted.
7. **`LIMIT` / `OFFSET`**: The number of returned rows is restricted.

### Supplemental Insights:

#### 1. Alias Availability (Why it matters)
Understanding this execution order explains why column aliases defined in the `SELECT` clause cannot be used in the `WHERE` clause (since `WHERE` executes before `SELECT`), but can often be used in `ORDER BY`.

**Example of Failure:**
```sql
SELECT first_name || ' ' || last_name AS full_name
FROM employees
WHERE full_name = 'John Doe'; -- ERROR: full_name is not available yet!
```
*Why it fails:* The database checks the `WHERE` condition before the `SELECT` clause creates the `full_name` alias.

**Example of Success:**
```sql
SELECT first_name || ' ' || last_name AS full_name
FROM employees
ORDER BY full_name; -- Works!
```
*Why it works:* `ORDER BY` is executed after `SELECT`, so the alias is available.

#### 2. Preparing for Subqueries: Solving Alias Issues
When filtering by a calculated alias (like `full_name`) is required, it cannot be used in the `WHERE` clause directly. One effective solution to this problem is using a **Subquery** (a query nested inside another query).

This approach allows the alias to be defined in an inner query and subsequently used in the `WHERE` clause of the outer query. This topic will be covered in detail later in this lecture.

**Example using Subquery:**
```sql
SELECT *
FROM (
    SELECT first_name || ' ' || last_name AS full_name
    FROM employees
) AS name_subquery
WHERE full_name = 'John Doe'; -- Now 'full_name' is available!
```
*Brief Explanation:* The inner query (inside the parentheses) runs first, creating the `full_name` alias. The outer query then treats the results of the inner query as a temporary table (aliased as `name_subquery`) and can successfully filter by `full_name` in its `WHERE` clause.

#### 3. Performance Implication
Filtering data as early as possible (in the `WHERE` clause or `JOIN` conditions) minimizes the volume of data processed in subsequent stages like grouping and sorting, drastically improving query performance.

In [ ]:
# Install Faker library for generating mock data
%pip install Faker --extra-index-url https://pypi.org/simple

### Hands-On Lab Part 2: Impact of Logical Execution Order

To demonstrate how understanding the logical execution order affects performance, a comparison can be made between two queries that return the identical result but operate differently behind the scenes.

#### Database Schema and ER Diagram

The demonstration utilizes a dedicated test database with two tables: `users` and `orders`. The relationship is one-to-many (one user can have multiple orders).

> [!NOTE]
> Execute the following code cell to render the Entity-Relationship Diagram (ERD) visually.

In [ ]:
import base64
from IPython.display import Image, display


def render_mermaid(graph_code):
    """
    Renders Mermaid diagrams into images using the mermaid.ink web API.
    Zero-dependency solution (requires no pip packages).
    """
    graph_code = graph_code.strip()

    graphbytes = graph_code.encode("utf8")
    base64_bytes = base64.b64encode(graphbytes)
    base64_string = base64_bytes.decode("ascii")

    image_url = f"https://mermaid.ink/img/{base64_string}"
    display(Image(url=image_url))


# ER Diagram for Hands-On Lab Part 2
erd_code = """
erDiagram
    student_users ||--o{ student_orders : "places"
    student_users {
        int id PK
        varchar name
        date signup_date
    }
    student_orders {
        int id PK
        int user_id FK
        decimal total_value
        date order_date
    }
"""

render_mermaid(erd_code)

**Schema Details**:
- **`student_users`**: Stores user profile information.
  - `id`: Unique identifier (Auto-increment).
  - `name`: Name of the user.
  - `signup_date`: The date the user registered.
- **`student_orders`**: Stores order transactions.
  - `id`: Unique identifier (Auto-increment).
  - `user_id`: References `student_users.id`.
  - `total_value`: Monetary value of the order.
  - `order_date`: The date the order was placed.

#### The Scenario
The objective is to find the total order value for users who signed up after a specific date ('2026-01-01').

#### The Queries

**Query 1**
```sql
SELECT u.id, u.name, SUM(o.total_value) as user_total
FROM student_users u
JOIN student_orders o ON u.id = o.user_id
GROUP BY u.id, u.name, u.signup_date
HAVING u.signup_date >= '2026-01-01';
```

**Query 2**
```sql
SELECT u.id, u.name, SUM(o.total_value) as user_total
FROM student_users u
JOIN student_orders o ON u.id = o.user_id
WHERE u.signup_date >= '2026-01-01'
GROUP BY u.id, u.name;
```

The following code creates the database, seeds it with thousands of records, and measures the execution time of both approaches.

In [ ]:
import time
import mysql.connector
import os
import random
from datetime import datetime, timedelta

# Database connection configuration
db_config = {
    "host": os.environ.get("MYSQL_HOST", "localhost"),
    "user": os.environ.get("MYSQL_USER", "student"),
    "password": os.environ.get("MYSQL_PASSWORD", "root"),
}


def run_demo():
    try:
        conn = mysql.connector.connect(**db_config)
        cursor = conn.cursor()

        # Changed to match the privilege pattern 'student_tasks.*'
        db_name = "student_tasks"
        print(f"Setting up test database '{db_name}'...")
        cursor.execute(f"CREATE DATABASE IF NOT EXISTS {db_name}")
        cursor.execute(f"USE {db_name}")

        # Create tables with student_ prefix
        cursor.execute("DROP TABLE IF EXISTS student_orders")
        cursor.execute("DROP TABLE IF EXISTS student_users")

        cursor.execute("""
            CREATE TABLE student_users (
                id INT AUTO_INCREMENT PRIMARY KEY,
                name VARCHAR(100),
                signup_date DATE
            )
        """)
        cursor.execute("""
            CREATE TABLE student_orders (
                id INT AUTO_INCREMENT PRIMARY KEY,
                user_id INT,
                total_value DECIMAL(10,2),
                order_date DATE,
                FOREIGN KEY (user_id) REFERENCES student_users(id)
            )
        """)

        # Data Generation
        print("Seeding users data WITHOUT Faker (using simple loops)...")
        users_data = []
        start_date = datetime(2025, 1, 1)
        for i in range(5000):
            name = f"User_{i}"  # Generic name
            days_to_add = random.randint(0, 500)
            signup_date = (start_date + timedelta(days=days_to_add)).strftime(
                "%Y-%m-%d"
            )
            users_data.append((name, signup_date))

        cursor.executemany(
            "INSERT INTO student_users (name, signup_date) VALUES (%s, %s)", users_data
        )
        conn.commit()

        cursor.execute("SELECT id FROM student_users")
        user_ids = [row[0] for row in cursor.fetchall()]

        print("Seeding orders data WITH Faker...")
        from faker import Faker

        fake = Faker()

        orders_data = []
        for i in range(20000):
            user_id = random.choice(user_ids)
            total_value = float(
                fake.pydecimal(left_digits=3, right_digits=2, positive=True)
            )
            order_date = fake.date_between(
                start_date=start_date, end_date="today"
            ).strftime("%Y-%m-%d")
            orders_data.append((user_id, total_value, order_date))

        cursor.executemany(
            "INSERT INTO student_orders (user_id, total_value, order_date) VALUES (%s, %s, %s)",
            orders_data,
        )
        conn.commit()
        print("Seeding completed.")

        print("\n--- Running Performance Demonstration ---")
        # 1. Query 1
        query1_sql = """
        SELECT u.id, u.name, SUM(o.total_value) as user_total
        FROM student_users u
        JOIN student_orders o ON u.id = o.user_id
        GROUP BY u.id, u.name, u.signup_date
        HAVING u.signup_date >= '2026-01-01'
        """
        start_time = time.time()
        cursor.execute(query1_sql)
        cursor.fetchall()
        query1_time = time.time() - start_time
        print(f"Query 1 completed in: {query1_time:.6f} seconds")

        # 2. Query 2
        query2_sql = """
        SELECT u.id, u.name, SUM(o.total_value) as user_total
        FROM student_users u
        JOIN student_orders o ON u.id = o.user_id
        WHERE u.signup_date >= '2026-01-01'
        GROUP BY u.id, u.name
        """
        start_time = time.time()
        cursor.execute(query2_sql)
        cursor.fetchall()
        query2_time = time.time() - start_time
        print(f"Query 2 completed in: {query2_time:.6f} seconds")
        print(
            f"Performance Difference: {((query1_time - query2_time) / query1_time) * 100:.2f}% difference"
        )

        cursor.close()
        conn.close()
    except Exception as e:
        print(f"Error occurred: {e}")


run_demo()

#### Analysis of Results: Why Query 2 Outperforms Query 1

After executing both queries, a significant difference in execution time should be observed. Here is the explanation of why one approach outperforms the other based on the **Logical Execution Order**:

1. **Query 1 (Inefficient)**: Uses `HAVING` to filter by date. Since `HAVING` executes *after* `GROUP BY`, the database must fetch all records, join them, and perform the grouping operation for *all* users before it can filter out those who signed up before '2026-01-01'. This wastes CPU and memory on data that is eventually discarded.
2. **Query 2 (Efficient)**: Uses `WHERE` to filter by date. Since `WHERE` executes *before* `GROUP BY`, the database discards irrelevant user records immediately after the join (or even before, depending on the optimizer). The grouping operation is then performed only on the subset of data that actually meets the criteria, resulting in a much faster execution time.

**Key Takeaway**: Always filter data as early as possible in the query lifecycle (ideally in the `WHERE` clause) to reduce the workload for subsequent operations like grouping and sorting.

## Subqueries (Nested Queries)

A subquery is a query nested inside another query. Subqueries can be used in various clauses, including `WHERE`, `FROM`, and `SELECT`. They allow for solving multi-step data retrieval problems in a single statement.

### 1. Subqueries in the `WHERE` Clause
Used to filter rows based on the result of another query.

*Example*: Find all employees whose salary is greater than the average salary.
```sql
SELECT name, salary 
FROM employees 
WHERE salary > (SELECT AVG(salary) FROM employees);
```

### 2. Subqueries in the `FROM` Clause (Derived Tables)
The result of the subquery is treated as a temporary table that can be queried.

*Example*: Find the department with the highest total salary.
```sql
SELECT department_id, total_salary
FROM (
    SELECT department_id, SUM(salary) AS total_salary
    FROM employees
    GROUP BY department_id
) AS dept_salaries
ORDER BY total_salary DESC
LIMIT 1;
```

### 3. Subqueries in the `SELECT` Clause (Scalar Subqueries)
Returns a single value for each row processed by the outer query.

*Example*: List employees and the average salary of their respective department.
```sql
SELECT name, salary,
       (SELECT AVG(salary) FROM employees e2 WHERE e2.department_id = e1.department_id) AS dept_avg
FROM employees e1;
```

### Supplemental Insights:
- **Correlated vs. Non-Correlated**: A non-correlated subquery (like the one in the `WHERE` example) can be executed independently of the outer query. A correlated subquery (like the one in the `SELECT` example) references columns from the outer query and is executed once for each row candidate in the outer query, which can be resource-intensive and should be avoided if a `JOIN` alternative exists.